# 春第 2 回｜表データと条件抽出

ボストン（BOS）行きの朝の便が月全体にはあっても，出かける日の 8 時台にあるとは限りません。2024 年 1 月の運航記録から行先・日付・出発予定時刻を重ねて便を選び，便が見つからないときは時間帯を広げて調べます。

最初に本文の例を一緒に実行し，後半では自分で条件を選んで約 20 分の実習に取り組みます。

## 準備

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

最初のセルはファイルの置き場所に対応する準備です。そのまま実行してください。出力の `32347` は，この CSV に入った予定便の総数です。この表は当時の運航記録であり，現在の時刻表ではありません。

In [ ]:
from pathlib import Path

import pandas as pd

file_name = "nyc-flights-2024-01.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

flights = pd.read_csv(csv_path)
len(flights)

## 本文と一緒に実行

各節で選びたい便を言葉で確かめ，コードを上から順に実行します。図の便数と，条件を重ねて選んだ表の行数がどう対応するかを読みます。

### 朝のボストン行きを図で見る

2024 年 1 月のボストン（BOS）行き予定便を，出発空港と 7〜9 時台で分けた図です。8 時台はどの空港に何便あるでしょうか。まず棒を読み，後で行先と時刻を元の表で選んで同じ件数になるか確かめます。

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt
import pandas as pd

morning_records = flights
morning_bos = morning_records.loc[morning_records["destination"] == "BOS"]
# 時刻コードの下 2 桁を除き，何時台かを得る。
morning_hours = morning_bos["scheduled_departure"] // 100
morning_counts = pd.crosstab(morning_bos["origin"], morning_hours)
morning_counts = morning_counts.reindex(
    index=["EWR", "JFK", "LGA"], columns=[7, 8, 9], fill_value=0
)

morning_figure, (ewr_panel, jfk_panel, lga_panel) = plt.subplots(
    1, 3, sharey=True, layout="constrained"
)
ewr_bars = ewr_panel.bar([7, 8, 9], morning_counts.loc["EWR"], color="#286b70")
jfk_bars = jfk_panel.bar([7, 8, 9], morning_counts.loc["JFK"], color="#286b70")
lga_bars = lga_panel.bar([7, 8, 9], morning_counts.loc["LGA"], color="#286b70")
ewr_panel.bar_label(ewr_bars, padding=3)
jfk_panel.bar_label(jfk_bars, padding=3)
lga_panel.bar_label(lga_bars, padding=3)
ewr_panel.set(title="EWR", xticks=[7, 8, 9], ylim=(0, 55), ylabel="Scheduled flights")
jfk_panel.set(title="JFK", xticks=[7, 8, 9])
lga_panel.set(title="LGA", xticks=[7, 8, 9])
morning_figure.supxlabel("Scheduled departure hour (local time)")
plt.show()

元の表は 1 行が 1 予定便です。行先や時刻を読める列だけを表示すると，表の行数と列数はそれぞれどうなるでしょうか。便を選ぶ前に，表示する列を減らすことと，行を絞ることを区別します。

In [ ]:
import pandas as pd

columns = ["date", "origin", "destination", "scheduled_departure", "arrival_delay"]
flights[columns].head(3)

In [ ]:
print(flights.shape)
print(flights[columns].shape)

### 行先を BOS に絞る

行先が BOS の便に `True` を付け，その数を確かめます。次に同じ判定を `.loc` に渡して便の行を取り出します。`True` の数と表の行数が一致するか，表示された行の `destination` が BOS かを照合します。

In [ ]:
destinations = flights["destination"]
destinations.head(3)

In [ ]:
target_destination = "BOS"
is_destination = destinations == target_destination
print(is_destination.sum(), len(is_destination))

In [ ]:
route_flights = flights.loc[is_destination]
print(len(route_flights))
route_flights[columns].head(3)

In [ ]:
example_rows = [20, 24]
route_flights.loc[example_rows, columns]

### 8 時台の始まりと終わり

8 時台には 8:00 の便を含め，9:00 の便は含めません。予定時刻を表す整数 `800` と `900` を境界に，二つの比較がそれぞれどの便を選ぶかを見ます。

In [ ]:
departure = route_flights["scheduled_departure"]
at_or_after_eight = departure >= 800
before_nine = departure < 900
print(at_or_after_eight.sum(), before_nine.sum())

In [ ]:
# 提供した作図セル。そのまま実行します。
range_figure, range_axis = plt.subplots(layout="constrained")
range_axis.hlines([2, 1, 0], 445, 575, color="#cccccc", linewidth=1)
range_axis.axvline(480, color="#777777", linestyle=":", linewidth=1)
range_axis.axvline(540, color="#777777", linestyle=":", linewidth=1)
range_axis.hlines([2, 1, 0], [480, 445, 480], [575, 540, 540],
                  color="#286b70", linewidth=4)
range_axis.scatter([480, 480], [2, 0], color="#286b70", s=45, zorder=3)
range_axis.scatter([540, 540], [1, 0], facecolors="white", edgecolors="#286b70",
                   linewidths=1.5, s=45, zorder=3)
range_axis.set(
    xlim=(445, 575), ylim=(-0.5, 2.5), xticks=[450, 480, 510, 540, 570],
    xticklabels=["7:30", "8:00", "8:30", "9:00", "9:30"],
    yticks=[2, 1, 0], yticklabels=[">= 8:00", "< 9:00", "Both"],
    xlabel="Scheduled departure (local time)",
)
range_axis.spines[["top", "left", "right"]].set_visible(False)
range_axis.tick_params(axis="y", length=0)
plt.show()

8:00 以上と 9:00 未満を同時に満たす BOS 行きの便を選びます。選んだ表を出発空港ごとに数え，冒頭の 8 時台の棒と一致するか確かめます。どちらか一方の条件だけでは余分な便が入る理由も考えてください。

In [ ]:
in_hour = at_or_after_eight & before_nine
hour_flights = route_flights.loc[in_hour]
print(len(hour_flights))
hour_flights[columns].head(3)

In [ ]:
origins = hour_flights["origin"]
print("EWR", (origins == "EWR").sum())
print("JFK", (origins == "JFK").sum())
print("LGA", (origins == "LGA").sum())

### 日付を選び，調べる時刻を広げる

月全体の 8 時台の便数からは，特定の日の予定を判断できません。1 月 14 日と 7 日の行を選び，それぞれ何便あるか，どの空港から何時に出る予定だったかを読みます。

In [ ]:
selected_day = "2024-01-14"
on_day = hour_flights["date"] == selected_day
day_at_eight = hour_flights.loc[on_day]
print(len(day_at_eight))
day_at_eight[columns]

In [ ]:
comparison_day = "2024-01-07"
on_comparison_day = hour_flights["date"] == comparison_day
hour_flights.loc[on_comparison_day, ["date", "origin", "scheduled_departure"]]

1 月 14 日の 8 時台には BOS 行きがありませんでした。その日の全時刻を持つ表に戻り，7:00 以上 9:00 未満，次に 7:00 以上 10:00 未満へ範囲を広げます。各段階で加わった便の時刻が，動かした境界とどう対応するかを確かめます。

In [ ]:
selected_day = "2024-01-14"
day_flights = route_flights.loc[route_flights["date"] == selected_day]
day_departure = day_flights["scheduled_departure"]
print(len(day_flights))

In [ ]:
start_time = 700
end_time = 900
from_start = day_departure >= start_time
before_lecture_end = day_departure < end_time
through_eight = day_flights.loc[from_start & before_lecture_end]
print(len(through_eight))
through_eight[["origin", "scheduled_departure"]]

In [ ]:
end_time = 1000
before_lecture_end = day_departure < end_time
through_nine = day_flights.loc[from_start & before_lecture_end]
print(len(through_nine))
through_nine[["origin", "scheduled_departure"]]

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt

day_figure, day_axis = plt.subplots(layout="constrained")
day_axis.axvspan(480, 540, color="#e7edea")
day_axis.hlines([0, 1, 2], 415, 605, color="#cccccc", linewidth=1)
airport_positions = {"EWR": 2, "JFK": 1, "LGA": 0}
window_times = through_nine["scheduled_departure"]
window_minutes = window_times // 100 * 60 + window_times % 100
window_airports = through_nine["origin"].map(airport_positions)
day_axis.scatter(window_minutes, window_airports, color="#286b70", s=43, zorder=3)
# 点のそばに，対応する時刻を表示する。
for minute, position in zip(window_minutes, window_airports):
    day_axis.annotate(
        f"{minute // 60}:{minute % 60:02d}", (minute, position),
        xytext=(0, 7), textcoords="offset points", ha="center",
    )
day_axis.set(
    xlim=(409, 611), ylim=(-0.4, 2.45), xticks=[420, 480, 540, 600],
    xticklabels=["7:00", "8:00", "9:00", "10:00"],
    yticks=[0, 1, 2], yticklabels=["LGA", "JFK", "EWR"],
    xlabel="Scheduled departure (local time)",
)
day_axis.spines[["top", "left", "right"]].set_visible(False)
day_axis.tick_params(axis="y", length=0)
plt.show()

今度は全月の BOS 行きから時間帯を先に選び，その後で 1 月 14 日を選びます。先に日付を選んだ場合と，同じ 5 便にたどり着くでしょうか。途中の表がそれぞれ何便を含むかにも注目します。

In [ ]:
month_in_window = (departure >= start_time) & (departure < end_time)
month_morning = route_flights.loc[month_in_window]
morning_on_day = month_morning.loc[month_morning["date"] == selected_day]
print(len(month_morning), len(morning_on_day))
morning_on_day[["origin", "scheduled_departure"]]

### 到着の記録で 15 分を境に数える

冒頭の図は出発予定のある便を数えました。同じ 8 時台の表で，到着のずれを［15 分未満］［15 分以上］［記録なし］に分けます。三つの件数を足して予定便数と照合し，どの便には到着のずれを計算できないか調べます。

In [ ]:
print(len(hour_flights))
delay = hour_flights["arrival_delay"]
under_limit = delay < 15
at_or_over_limit = delay >= 15
print(under_limit.sum(), at_or_over_limit.sum())

In [ ]:
delay_columns = ["date", "origin", "scheduled_departure", "arrival_delay"]
hour_flights.loc[at_or_over_limit, delay_columns].head(3)

In [ ]:
unknown_delay = delay.isna()
print(under_limit.sum(), at_or_over_limit.sum(), unknown_delay.sum(), len(hour_flights))

到着の遅れが空欄の行を開き，`cancelled` と `diverted` を見ます。最後に，［15 分未満］以外を一つの表にすると，どんな行が一緒に入るかを確かめます。

In [ ]:
result_columns = ["date", "origin", "scheduled_departure", "arrival_delay", "cancelled", "diverted"]
hour_flights.loc[unknown_delay, result_columns]

In [ ]:
review_arrivals = hour_flights.loc[~under_limit]
print(len(review_arrivals))

## Jupyter 実習（約 20 分）

2024 年 1 月 14 日に，ニューヨーク周辺から朝出る予定だった便を調べます。本文の BOS 行きから，シカゴ・オヘア（ORD）またはアトランタ（ATL）へ行先を変えます。8 時台に便があるかを基準に，時間帯を一方へ広げると新たにどの便が見つかるか調べます。元の 8 時台の表を残し，広げた後の表と比べてください。

共通課題は 1〜3 の約 20 分です。目標は件数を出すだけでなく，**どの条件を変えると，どんな便が新しく見つかったかを説明すること**です。末尾の到着記録の課題は任意です。

### 1. 行先と基準の表（目安 4 分）

ORD または ATL を選びます。下の `target_destination` に選んだコードを指定してください。日付は 1 月 14 日のままにします。

先に行先を選び，そこから日付を選びます。この時点では時間を絞りません。あとで時間帯を広げるために，その日の全時刻を含む `day_flights` を残しておきます。

In [ ]:
target_destination = "ORD"
selected_day = "2024-01-14"

is_destination = flights["destination"] == target_destination
route_flights = flights.loc[is_destination]
on_day = route_flights["date"] == selected_day
day_flights = route_flights.loc[on_day]

columns = ["date", "origin", "destination", "carrier", "scheduled_departure"]
day_flights[columns].head()

次に 8:00 以上・9:00 未満の予定便を選びます。`departure` は **その日の全時刻** の出発予定時刻の列です。`baseline` は，比較の基準として残す 8 時台の表です。

表示された便の出発空港と予定時刻を読んでください。`800` は 8:00 です。便が表示される順番は，必ずしも時刻順ではありません。

`carrier` は航空会社のコードです。同じ空港・予定時刻でも，航空会社が異なる行があります。時刻だけで同じ便と判断せず，元の行を一行ずつ数えます。

In [ ]:
departure = day_flights["scheduled_departure"]
in_eight = (departure >= 800) & (departure < 900)
baseline = day_flights.loc[in_eight]

print(len(baseline))
baseline[columns]

**ここに記録します。**

- 選んだ行先：
- 8 時台の予定便数：
- 表にあった出発空港と時刻：

### 2. 時間帯を広げる（目安 10 分）

次のどちらかを選びます。元の 8 時台の表には入らず，変更後に加わるのは何時台の便かを，実行前に予想してください。

- **7 時台も含める：** 下限を `700` に変え，上限は `900` のままにする。
- **9 時台も含める：** 下限は `800` のままにして，上限を `1000` に変える。

どちらを選んでも，行先と日付はそのままです。一度に両方の境界を変えず，一つの変更と増えた便を対応させます。

**自分が調べる時間帯と，増えると予想した記録：**

（ここに書く）

In [ ]:
start_time = 800
end_time = 900

# 上の二つの値のうち，選んだ境界を一つだけ変更する。
at_or_after_start = departure >= start_time

下のセルの最初の行で，`before_end =` の右側に「出発予定時刻が上限より前」という比較を書いてください。本文の「時間帯を選ぶ」が手掛かりになります。

次に，**下の 5 行すべての先頭にある `#` を外して**実行します。`#` があるままだとコメントなので，計算は行われません。2 行目以降は書き写す必要はありません。入力した比較がどこに使われるかを読みながら実行してください。

`expanded` は変更後の表です。`baseline` は上書きせずに残します。

In [ ]:
# before_end =
# in_window = at_or_after_start & before_end
# expanded = day_flights.loc[in_window]
# print(len(expanded))
# expanded[columns]

<details>
<summary>比較式を確認したいとき</summary>

`departure` の値が `end_time` より小さいかを比べます。

```python
before_end = departure < end_time
```

上限を含めないので `<` を使います。たとえば上限が `1000` なら 9:59 は入りますが，10:00 は入りません。

`NameError` が出た場合は，`before_end` の行の `#` を外したか，その前の準備・行先・時間帯のセルを実行したかを確かめます。下限や上限を変更した後は，比較を作るセルと抽出するセルをどちらも再実行してください。

</details>

上の変更後の表と，次の基準の表を比べます。新しく見つかった便の予定時刻が，自分で加えた時間帯に入るかを読んでください。

In [ ]:
print(len(baseline))
baseline[columns]

**比較の記録**

- 変更前の条件と件数：
- 変更後の条件と件数：
- 新しく含まれた便の出発空港と予定時刻：

<details>
<summary>結果の件数を照合する</summary>

1 月 14 日の記録について，次の件数になります。件数だけが同じでも同じ便を選んだとは限らないので，出発空港と予定時刻も読んでください。

| 行先 | 8:00 以上・9:00 未満 | 7:00 以上・9:00 未満 | 8:00 以上・10:00 未満 |
|---|---:|---:|---:|
| ORD | 2 | 5 | 5 |
| ATL | 2 | 6 | 5 |

基準の件数のままの場合は，下限か上限を変えた後に，判定と抽出も実行し直したかを確かめます。時間を広げるときは，`baseline` の中だけを探すのではなく，全時刻を持つ `day_flights` から選び直します。

</details>

### 3. 見つかったことを伝える（目安 6 分）

自分が選んだ時間帯について，次の三点を短い段落にまとめてください。件数の増減だけでなく，動かした境界と新しく入った便の時刻を結び付けます。

1. いつの，どこへ向かう予定便を調べ，時間のどちらの境界を変えたか。
2. 何便から何便になり，どの空港・時刻の便が新しく見つかったか。
3. 変更前の `baseline` だけから選び直しても，新しい時間帯の便が見つからないのはなぜか。

**説明を書く欄：**

（ここに書く）

今回の表から，その便が実際にその時刻に出発したかも分かるでしょうか。`scheduled_departure` が何を記録した列かを踏まえ，最後に一文添えてください。

提出前にノートを保存し，最初から実行して，自分の条件と説明に合った結果になるかを確かめます。提出するのは，選んだ条件・実行結果・上の説明が入ったノートブックです。

### 発展：到着記録も調べる（任意）

時間に余裕があれば，自分が作った `expanded` について，次を調べてください。共通課題の提出に必要な追加作業ではありません。

- 到着の遅れが 15 分未満，15 分以上，空欄の便はそれぞれ何便か。
- 三つの件数を足すと `len(expanded)` と一致するか。
- 「15 分以上」と「15 分未満ではなかった」を同じ件数だと考えてよいか。今回一致していても，いつも一致すると言えるか。

本文の「予定と到着の記録」を参考に，必要なコードを下のセルへ書きます。

In [ ]:
# 任意：expanded の arrival_delay を取り出して調べる。